<a href="https://colab.research.google.com/github/anthony-yt/deepwater-image-segmentation-classical-quantum/blob/feat%2Fsecond-annotation-ariana/notebooks/2_4_second_annotation_ariana.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 5.6 - Segunda anotacion de una muestra (Ariana)

**Objetivo.** Aportar la segunda anotacion independiente que permite estimar la concordancia entre
anotadores, y responder si el criterio 5.0 deja decisiones abiertas.

**Asignacion.** `Dataset_Processed/Seleccion/seleccion_5_6.csv`, filas con `persona = Ariana`:
6 frames de Video2 y 6 de Video3, **12 mascaras**.

**Entregable.** Las 12 mascaras en el formato de la task 5.0, en
`Dataset_Processed/Segunda_Anotacion/Ariana/`, mas la lista de casos dudosos.

---

## Condicion de ciego

La anotacion debe hacerse **sin ver la mascara original**. Este notebook:

- escribe en `Segunda_Anotacion/Ariana/`, nunca en `Test_Set_Curado/`, y lo comprueba con un
  assert antes de guardar nada;
- no carga ni muestra en ningun momento una mascara de `Test_Set_Curado`;
- avisa si algun frame asignado ya tiene mascara curada, para que sepas que la condicion de ciego
  depende de que no la abras por otra via.

En el momento de escribir este notebook, `Test_Set_Curado` contiene Video1 y Video6 solamente, asi
que para los videos 2 y 3 no existe aun una primera anotacion que pudiera contaminar la segunda.

## Criterio

El de la task 5.0 de Anthony:

- **estructura**: todo objeto solido de origen humano (cascos, rejillas, helices, tuberias);
- **fondo**: el agua y el lecho marino;
- **casos ambiguos** (cables, objetos sueltos, sombras): se asignan a **fondo** y se registran.

Formato de salida, tambien de la task 5.0: PNG de un canal, valores 0 y 255, mismo tamano que el
frame, nombre `{frame}_mask.png`, sin antialiasing.

Cada frame en el que dudes se anota en `DUDAS` con el motivo: esa lista es la mitad del entregable,
porque es la que dice si el criterio 5.0 deja decisiones abiertas.

## Procedimiento por frame

1. `start(siguiente())` carga el frame y abre el lienzo.
2. Click izquierdo donde **hay** estructura, click derecho (o shift+click) donde **no** la hay.
3. `preview()` muestra las tres mascaras candidatas de SAM.
4. `save(i)` guarda la candidata `i`. `save(i, duda="...")` la guarda y registra la duda.
   `guardar_vacia()` guarda una mascara toda a cero si el frame no tiene estructura.
5. `resumen()` para ver lo que falta.

**Herramienta.** La mecanica del lienzo y del guardado sigue la del cuaderno de etiquetado asistido
de la task 5.8 (`2_2_assisted_labelling.ipynb`, rama `feat/labelling-evaluation-tools`). Cuando esa
rama se mergee y se extraiga a un modulo compartido, este notebook deberia importarlo en vez de
repetir la mecanica.

## 1. Entorno

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("segment_anything") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/facebookresearch/segment-anything.git"], check=True)

import base64
import os
import time
import urllib.request
import uuid
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, JSON, display
from PIL import Image

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, output
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/Under_Water_Dataset")
else:
    DRIVE_ROOT = Path(os.environ.get("DW_DATA_ROOT", "Under_Water_Dataset")).resolve()

PROCESSED = DRIVE_ROOT / "Dataset_Processed"
FRAMES_DIR = PROCESSED / "Frames_CLAHE"
SELECTION_CSV = PROCESSED / "Seleccion" / "seleccion_5_6.csv"
CURATED_DIR = PROCESSED / "Test_Set_Curado"

ANNOTATOR = "Ariana"
OUTPUT_DIR = PROCESSED / "Segunda_Anotacion" / ANNOTATOR
LOG_CSV = PROCESSED / "Registro_Etiquetado" / f"dudas_5_6_{ANNOTATOR}.csv"

# La condicion de ciego depende de esto: nunca se escribe sobre el conjunto de referencia.
assert OUTPUT_DIR.resolve() != CURATED_DIR.resolve(), "la salida no puede ser Test_Set_Curado"
assert "Segunda_Anotacion" in str(OUTPUT_DIR), "la salida debe estar en Segunda_Anotacion"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_CSV.parent.mkdir(parents=True, exist_ok=True)

print("frames   :", FRAMES_DIR)
print("asignacion:", SELECTION_CSV)
print("salida   :", OUTPUT_DIR)
print("dudas    :", LOG_CSV)

In [ ]:
CKPT = DRIVE_ROOT / "checkpoints" / "sam_vit_b_01ec64.pth"
CKPT.parent.mkdir(parents=True, exist_ok=True)
if not CKPT.exists():
    print("descargando SAM ViT-B...")
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth", CKPT)

from segment_anything import SamPredictor, sam_model_registry

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
predictor = SamPredictor(sam_model_registry["vit_b"](checkpoint=str(CKPT)).to(device=DEVICE))
print("SAM vit_b en", DEVICE)

## 2. Asignacion

In [ ]:
seleccion = pd.read_csv(SELECTION_CSV)
mios = seleccion[seleccion.persona.str.strip().str.casefold() == ANNOTATOR.casefold()]
FRAME_IDS = list(mios.frame_id)

print(f"{len(FRAME_IDS)} frames asignados a {ANNOTATOR}")
print(mios.groupby("video").frame_id.count().to_string())
print()
for fid in FRAME_IDS:
    existe_frame = any((FRAMES_DIR / f"{fid}{e}").exists() for e in (".jpg", ".jpeg", ".png"))
    ya_curado = (CURATED_DIR / f"{fid}_mask.png").exists()
    marca = "" if existe_frame else "  <-- NO ENCUENTRO EL FRAME"
    if ya_curado:
        marca += "  <-- ya existe mascara curada: no la abras, romperia el ciego"
    print(f"  {fid}{marca}")

faltantes = [f for f in FRAME_IDS
             if not any((FRAMES_DIR / f"{f}{e}").exists() for e in (".jpg", ".jpeg", ".png"))]
assert not faltantes, f"frames no encontrados en {FRAMES_DIR}: {faltantes}"

## 3. Lienzo de anotacion

In [ ]:
POINTS = []
STATE = {"frame": None, "image": None, "t0": None, "candidates": None, "scores": None}
DUDAS = (pd.read_csv(LOG_CSV).fillna("").to_dict("records") if LOG_CSV.exists() else [])


def hechas():
    return {p.stem.replace("_mask", "") for p in OUTPUT_DIR.glob("*_mask.png")}


def pendientes():
    return [f for f in FRAME_IDS if f not in hechas()]


def siguiente():
    falta = pendientes()
    print(f"{len(falta)} pendientes de {len(FRAME_IDS)}")
    return falta[0] if falta else FRAME_IDS[-1]


def frame_path(frame_id):
    for ext in (".jpg", ".jpeg", ".png"):
        p = FRAMES_DIR / f"{frame_id}{ext}"
        if p.exists():
            return p
    raise FileNotFoundError(frame_id)


if IN_COLAB:
    def _add(x, y, label):
        POINTS.append((float(x), float(y), int(label)))
        return JSON({"n": len(POINTS)})

    def _undo():
        if POINTS:
            POINTS.pop()
        return JSON({"n": len(POINTS)})

    def _clear():
        POINTS.clear()
        return JSON({"n": 0})

    output.register_callback("dw.add", _add)
    output.register_callback("dw.undo", _undo)
    output.register_callback("dw.clear", _clear)


def canvas(rgb):
    uid = uuid.uuid4().hex[:8]
    buf = Image.fromarray(rgb)
    import io
    raw = io.BytesIO()
    buf.save(raw, format="PNG")
    b64 = base64.b64encode(raw.getvalue()).decode()
    h, w = rgb.shape[:2]
    display(HTML(f"""
<button id="u{uid}">Deshacer</button> <button id="c{uid}">Limpiar</button>
&nbsp;click izquierdo: estructura &middot; click derecho o shift+click: fondo<br>
<canvas id="cv{uid}" width="{w}" height="{h}" style="cursor:crosshair;max-width:100%"></canvas>
<script>
const cv = document.getElementById("cv{uid}"), ctx = cv.getContext("2d"), img = new Image();
let pts = [];
img.src = "data:image/png;base64,{b64}";
function draw() {{
  ctx.drawImage(img, 0, 0);
  pts.forEach(p => {{ ctx.beginPath(); ctx.arc(p[0], p[1], 6, 0, 6.3);
                     ctx.fillStyle = p[2] ? "#00ff00" : "#ff0000"; ctx.fill();
                     ctx.lineWidth = 2; ctx.strokeStyle = "#fff"; ctx.stroke(); }});
}}
img.onload = draw;
function add(e, label) {{
  const r = cv.getBoundingClientRect();
  const x = (e.clientX - r.left) * cv.width / r.width;
  const y = (e.clientY - r.top) * cv.height / r.height;
  pts.push([x, y, label]); draw();
  google.colab.kernel.invokeFunction("dw.add", [x, y, label], {{}});
}}
cv.addEventListener("click", e => add(e, e.shiftKey ? 0 : 1));
cv.addEventListener("contextmenu", e => {{ e.preventDefault(); add(e, 0); }});
document.getElementById("u{uid}").onclick = () => {{ pts.pop(); draw();
  google.colab.kernel.invokeFunction("dw.undo", [], {{}}); }};
document.getElementById("c{uid}").onclick = () => {{ pts = []; draw();
  google.colab.kernel.invokeFunction("dw.clear", [], {{}}); }};
</script>"""))


def start(frame_id):
    rgb = np.asarray(Image.open(frame_path(frame_id)).convert("RGB"))
    POINTS.clear()
    STATE.update(frame=frame_id, image=rgb, t0=time.time(), candidates=None, scores=None)
    predictor.set_image(rgb)
    print(f"{frame_id}  {rgb.shape[1]} x {rgb.shape[0]}")
    canvas(rgb)

In [ ]:
def preview():
    """Tres candidatas de SAM para los puntos marcados."""
    if not POINTS:
        raise RuntimeError("marca al menos un punto de estructura antes de llamar a preview()")
    pts = np.array([[x, y] for x, y, _ in POINTS], dtype=np.float32)
    labels = np.array([l for _, _, l in POINTS], dtype=np.int32)
    masks, scores, _ = predictor.predict(point_coords=pts, point_labels=labels,
                                         multimask_output=True)
    STATE["candidates"], STATE["scores"] = masks, scores

    rgb = STATE["image"]
    fig, axes = plt.subplots(1, len(masks) + 1, figsize=(4.4 * (len(masks) + 1), 4.0))
    axes[0].imshow(rgb)
    axes[0].scatter(pts[labels == 1][:, 0], pts[labels == 1][:, 1], c="lime", s=40,
                    edgecolors="white")
    if (labels == 0).any():
        axes[0].scatter(pts[labels == 0][:, 0], pts[labels == 0][:, 1], c="red", s=40,
                        edgecolors="white")
    axes[0].set_title("frame y puntos", fontsize=9)
    for i, (mask, score) in enumerate(zip(masks, scores)):
        overlay = rgb.copy()
        overlay[mask] = (0.45 * overlay[mask] + 0.55 * np.array([255, 40, 40])).astype(np.uint8)
        axes[i + 1].imshow(overlay)
        axes[i + 1].set_title(f"save({i})   score={score:.3f}\nestructura={mask.mean():.3f}",
                              fontsize=9)
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def _escribir(mask_bool, duda):
    """Guarda PNG de un canal 0/255 al tamano del frame, sin reescalar."""
    frame_id = STATE["frame"]
    rgb = STATE["image"]
    assert mask_bool.shape == rgb.shape[:2], \
        f"la mascara {mask_bool.shape} no coincide con el frame {rgb.shape[:2]}"

    # uint8 2D -> PIL infiere modo "L": un canal, sin paleta y sin antialiasing, porque no
    # se reescala en ningun momento.
    array = np.where(mask_bool, 255, 0).astype(np.uint8)
    destino = OUTPUT_DIR / f"{frame_id}_mask.png"
    assert OUTPUT_DIR.resolve() != CURATED_DIR.resolve(), "destino incorrecto"
    imagen = Image.fromarray(array)
    assert imagen.mode == "L", f"modo inesperado: {imagen.mode}"
    imagen.save(destino, optimize=True)

    DUDAS.append({"frame_id": frame_id, "anotador": ANNOTATOR,
                  "segundos": round(time.time() - STATE["t0"], 1),
                  "fraccion_estructura": round(float(mask_bool.mean()), 4),
                  "duda": duda})
    pd.DataFrame(DUDAS).to_csv(LOG_CSV, index=False)
    print(f"guardada {destino.name}  estructura={mask_bool.mean():.3f}"
          f"  {'| DUDA: ' + duda if duda else ''}")
    print(f"quedan {len(pendientes())} de {len(FRAME_IDS)}")


def save(i=0, duda=""):
    if STATE["candidates"] is None:
        raise RuntimeError("llama a preview() antes de save()")
    _escribir(STATE["candidates"][i].astype(bool), duda)


def guardar_vacia(duda="frame sin estructura visible"):
    """Mascara toda a cero: el frame no contiene estructura."""
    _escribir(np.zeros(STATE["image"].shape[:2], dtype=bool), duda)


def resumen():
    hecho = hechas() & set(FRAME_IDS)
    print(f"{len(hecho)} de {len(FRAME_IDS)} anotadas")
    if DUDAS:
        print()
        print(pd.DataFrame(DUDAS)[["frame_id", "segundos", "fraccion_estructura", "duda"]]
              .to_string(index=False))
    if pendientes():
        print("\npendientes:", pendientes())

## 4. Anotar

Ejecuta estas celdas en bucle, una vez por frame.

In [ ]:
start(siguiente())

In [ ]:
preview()

In [ ]:
# save(0)                                     elige la candidata 0
# save(1, duda="el cable del borde izquierdo: lo dejo en fondo segun el criterio 5.0")
# guardar_vacia()                               el frame no tiene estructura
save(0)

In [ ]:
resumen()

## 5. Verificacion del entregable

Comprueba que las 12 mascaras cumplen el formato de la task 5.0 antes de dar la tarea por cerrada.
Esta celda **no** abre `Test_Set_Curado`: solo valida el formato de lo producido.

In [ ]:
filas = []
for fid in FRAME_IDS:
    p = OUTPUT_DIR / f"{fid}_mask.png"
    if not p.exists():
        filas.append({"frame_id": fid, "existe": False})
        continue
    im = Image.open(p)
    arr = np.asarray(im)
    frame = Image.open(frame_path(fid))
    filas.append({
        "frame_id": fid, "existe": True, "modo": im.mode,
        "un_canal": arr.ndim == 2,
        "tamano_ok": im.size == frame.size,
        "valores": sorted(np.unique(arr).tolist()),
        "binaria_0_255": set(np.unique(arr).tolist()) <= {0, 255},
        "fraccion_estructura": round(float((arr > 127).mean()), 4),
    })
check = pd.DataFrame(filas)
print(check.to_string(index=False))

problemas = []
if not check.existe.all():
    problemas.append(f"faltan {int((~check.existe).sum())} mascaras")
if check.existe.all():
    if not check.modo.eq("L").all():
        problemas.append("alguna mascara no esta en modo L (un canal)")
    if not check.binaria_0_255.all():
        problemas.append("alguna mascara tiene valores distintos de 0 y 255")
    if not check.tamano_ok.all():
        problemas.append("alguna mascara no tiene el tamano del frame")

print()
if problemas:
    for p in problemas:
        print("PROBLEMA:", p)
else:
    print(f"las {len(check)} mascaras cumplen el formato de la task 5.0")
    print(f"fraccion de estructura media: {check.fraccion_estructura.mean():.4f}")
    print(check.groupby(check.frame_id.str.extract(r'(Video\d+)')[0])
          .fraccion_estructura.agg(["count", "mean", "min", "max"]).round(4).to_string())

## 6. Casos dudosos

La lista de dudas es la segunda mitad del entregable: es la que responde si el criterio 5.0 deja
decisiones abiertas. Se guarda en `Registro_Etiquetado/dudas_5_6_Ariana.csv` a medida que anotas.

Al terminar, resume aqui los tipos de duda que aparecieron y pasalos a Anthony para la task 5.6
(concordancia entre anotadores) y para el ajuste del criterio 5.0.

In [ ]:
if LOG_CSV.exists():
    dudas = pd.read_csv(LOG_CSV).fillna("")
    con_duda = dudas[dudas.duda != ""]
    print(f"{len(con_duda)} frames anotados con duda, de {len(dudas)}\n")
    for row in con_duda.itertuples():
        print(f"- {row.frame_id}: {row.duda}")
    print(f"\ntiempo total: {dudas.segundos.sum() / 60:.1f} min"
          f"  |  mediana por frame: {dudas.segundos.median():.0f} s")
else:
    print("todavia no hay registro")

### Resumen para Anthony

*Rellenar al terminar.*

- **Tipos de duda encontrados:**
- **Vacios del criterio 5.0 que conviene cerrar:**
- **Frames donde la decision cambiaria el IoU de forma apreciable:**